In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [23]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("AdvSparkApp") \
    .getOrCreate()


In [24]:
# Create DataFrame with 1,000,000 rows
df = spark.range(1, 1000000).withColumn("value", col("id") % 1000)

In [25]:
# Show first 100 rows
df.show(100)

+---+-----+
| id|value|
+---+-----+
|  1|    1|
|  2|    2|
|  3|    3|
|  4|    4|
|  5|    5|
|  6|    6|
|  7|    7|
|  8|    8|
|  9|    9|
| 10|   10|
| 11|   11|
| 12|   12|
| 13|   13|
| 14|   14|
| 15|   15|
| 16|   16|
| 17|   17|
| 18|   18|
| 19|   19|
| 20|   20|
| 21|   21|
| 22|   22|
| 23|   23|
| 24|   24|
| 25|   25|
| 26|   26|
| 27|   27|
| 28|   28|
| 29|   29|
| 30|   30|
| 31|   31|
| 32|   32|
| 33|   33|
| 34|   34|
| 35|   35|
| 36|   36|
| 37|   37|
| 38|   38|
| 39|   39|
| 40|   40|
| 41|   41|
| 42|   42|
| 43|   43|
| 44|   44|
| 45|   45|
| 46|   46|
| 47|   47|
| 48|   48|
| 49|   49|
| 50|   50|
| 51|   51|
| 52|   52|
| 53|   53|
| 54|   54|
| 55|   55|
| 56|   56|
| 57|   57|
| 58|   58|
| 59|   59|
| 60|   60|
| 61|   61|
| 62|   62|
| 63|   63|
| 64|   64|
| 65|   65|
| 66|   66|
| 67|   67|
| 68|   68|
| 69|   69|
| 70|   70|
| 71|   71|
| 72|   72|
| 73|   73|
| 74|   74|
| 75|   75|
| 76|   76|
| 77|   77|
| 78|   78|
| 79|   79|
| 80|   80|
| 81

In [26]:
# Check number of partitions before repartitioning
print("Before Repartition:", df.rdd.getNumPartitions())

Before Repartition: 2


In [27]:
# Repartition DataFrame
df_repartitioned = df.repartition(8)

In [28]:
# Check number of partitions after repartitioning
print("After Repartition:", df_repartitioned.rdd.getNumPartitions())

After Repartition: 8


In [29]:
# Coalesce partitions
df_coalesced = df_repartitioned.coalesce(4)

# Check number of partitions after coalescing
print("After Coalesce:", df_coalesced.rdd.getNumPartitions())

After Coalesce: 4


[Stage 11:=============================>                            (1 + 1) / 2]

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("AdvancedSparkApp") \
    .getOrCreate()

df = spark.range(0, 10000000) \
          .withColumn("value", col("id") % 1000)

26/06/13 06:49:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [31]:
optimized_df = df.filter(col("value") > 500) \
                 .filter(col("id") < 5000000) \
                 .select("id", "value")

optimized_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [id#71L, value#73L]
      +- InMemoryRelation [id#71L, value#73L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Project [id#0L, (id#0L % 1000) AS value#2L]
               +- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
                  +- *(1) Range (0, 10000000, step=1, splits=2)




In [32]:
import time

start_time = time.time()

count_uncached = optimized_df.count()  # Action triggers DAG

end_time = time.time()

print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

[Stage 12:=============================>                            (1 + 1) / 2]

1. Optimized Execution | Count: 2495000 | Time: 2.0289 seconds


In [33]:
optimized_df.cache()

26/06/13 06:49:38 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [35]:
def can_drive(age):
    if age>=18:
        return "can Drive"
    else:
        return "cannot Drive"

In [36]:
drive_udf = udf(can_drive,StringType())

In [37]:
data =[
    ("Ram",16),
    ("Shyam",20),
    ("Rita",25)
]

In [42]:
df = spark.createDataFrame(data,["Name","Age"])


In [43]:
df = df.withColumn("Driving_Status", drive_udf("Age"))

In [ ]:
df.show(20)

[Stage 16:>                                                         (0 + 1) / 1]

+-----+---+--------------+
| Name|Age|Driving_Status|
+-----+---+--------------+
|  Ram| 16|  cannot Drive|
|Shyam| 20|     can Drive|
| Rita| 25|     can Drive|
+-----+---+--------------+

